In [1]:
import pandas as pd

def load_and_prepare_behav(subj_num, behav_base_path):
    behav_path = f"{behav_base_path}/sub{subj_num:02d}_behav.tsv"
    behav_df = pd.read_csv(behav_path, sep='\t')
    
    # Create cumulative fmri_index per session (0 to 749)
    behav_df['fmri_index'] = None
    for session in behav_df['SESSION'].unique():
        idxs = behav_df[behav_df['SESSION'] == session].index
        behav_df.loc[idxs, 'fmri_index'] = range(len(idxs))
    behav_df['fmri_index'] = behav_df['fmri_index'].astype(int)
    
    print(f"Subject {subj_num}: Loaded behav with {len(behav_df)} rows")
    return behav_df


In [2]:
def get_shared_trials_by_phase(behav_df):
    """
    Filter for shared1000 images and classify trials into encoding/retrieval phases.
    
    Returns:
        - encoding_df: DataFrame for encoding trials
        - retrieval1_df: DataFrame for first retrieval
        - retrieval2_df: DataFrame for second retrieval (if present)
    """
    # Keep only images that have a repNum entry (i.e., from shared1000 set)
    shared_df = behav_df[behav_df["shared1000_repNum"].notna()].copy()
    shared_df["shared1000_repNum"] = shared_df["shared1000_repNum"].astype(int)

    # Sanity check: each image ID (73KID) should occur at least twice
    image_counts = shared_df["73KID"].value_counts()
    valid_ids = image_counts[image_counts >= 2].index
    shared_df = shared_df[shared_df["73KID"].isin(valid_ids)]

    # Label trial phase
    phase_map = {1: "encoding", 2: "retrieval1", 3: "retrieval2"}
    shared_df["PHASE"] = shared_df["shared1000_repNum"].map(phase_map)

    # Now split by phase
    encoding_df = shared_df[shared_df["PHASE"] == "encoding"]
    retrieval1_df = shared_df[shared_df["PHASE"] == "retrieval1"]
    retrieval2_df = shared_df[shared_df["PHASE"] == "retrieval2"]

    return encoding_df, retrieval1_df, retrieval2_df


In [3]:
def get_phase_maps_for_session(behav_df, session_num):
    behav_sess = behav_df[behav_df['SESSION'] == session_num]
    encoding_df, retrieval1_df, retrieval2_df = get_shared_trials_by_phase(behav_sess)

    enc_kids = set(encoding_df['73KID'])
    ret1_kids = set(retrieval1_df['73KID'])
    ret2_kids = set(retrieval2_df['73KID'])
    common_kids_sess = enc_kids & ret1_kids & ret2_kids

    enc_map = dict(zip(encoding_df['73KID'], encoding_df['fmri_index']))
    ret1_map = dict(zip(retrieval1_df['73KID'], retrieval1_df['fmri_index']))
    ret2_map = dict(zip(retrieval2_df['73KID'], retrieval2_df['fmri_index']))

    print(f"Session {session_num}: {len(common_kids_sess)} common trials across phases")
    return common_kids_sess, enc_map, ret1_map, ret2_map


In [4]:
import h5py
from sklearn.metrics.pairwise import cosine_similarity

def cosine_sim(vec1, vec2):
    return cosine_similarity(vec1.reshape(1, -1), vec2.reshape(1, -1))[0][0]

def run_subject_pipeline(subj_num, behav_base_path, h5_base_path):
    print(f"\n--- Running pipeline for Subject {subj_num} ---")
    
    behav_df = load_and_prepare_behav(subj_num, behav_base_path)
    h5_path = f"{h5_base_path}/subj{subj_num:02d}_all_mtl_sessions.h5"
    
    similarity_results = []
    
    with h5py.File(h5_path, 'r') as f:
        sessions = list(f.keys())
        print(f"Subject {subj_num}: Found {len(sessions)} sessions in HDF5")
        
        for session in sessions:
            session_num = int(session[-2:])
            
            common_kids, enc_map, ret1_map, ret2_map = get_phase_maps_for_session(behav_df, session_num)
            roi_names = list(f[session].keys())
            
            for roi in roi_names:
                roi_data = f[session][roi]
                
                for kid in common_kids:
                    try:
                        enc_idx = enc_map[kid]
                        ret1_idx = ret1_map[kid]
                        ret2_idx = ret2_map[kid]

                        enc_beta = roi_data[enc_idx, :]
                        ret1_beta = roi_data[ret1_idx, :]
                        ret2_beta = roi_data[ret2_idx, :]

                        sim_enc_ret1 = cosine_sim(enc_beta, ret1_beta)
                        sim_enc_ret2 = cosine_sim(enc_beta, ret2_beta)
                        sim_ret1_ret2 = cosine_sim(ret1_beta, ret2_beta)

                        similarity_results.append({
                            'SUBJECT': subj_num,
                            'SESSION': session_num,
                            '73KID': kid,
                            'ROI': roi,
                            'enc_ret1': sim_enc_ret1,
                            'enc_ret2': sim_enc_ret2,
                            'ret1_ret2': sim_ret1_ret2
                        })
                    except KeyError:
                        continue

    similarity_df = pd.DataFrame(similarity_results)
    print(f"Subject {subj_num}: Computed similarities for {len(similarity_df)} trials")
    
    # Save to CSV
    out_csv = f"{h5_base_path}/subj{subj_num:02d}_similarity_results.csv"
    similarity_df.to_csv(out_csv, index=False)
    print(f"Subject {subj_num}: Results saved to {out_csv}")
    
    return similarity_df


In [6]:
behav_base_path = "/home/jovyan/cache/memoryNSD"
h5_base_path = "/home/jovyan/cache/memoryNSD"

for subj in range(4,9):  # add all your subjects here
    df = run_subject_pipeline(subj, behav_base_path, h5_base_path)



--- Running pipeline for Subject 4 ---
Subject 4: Loaded behav with 22500 rows
Subject 4: Found 30 sessions in HDF5
Session 1: 2 common trials across phases
Session 2: 4 common trials across phases
Session 3: 2 common trials across phases
Session 4: 4 common trials across phases
Session 5: 1 common trials across phases
Session 6: 5 common trials across phases
Session 7: 2 common trials across phases
Session 8: 2 common trials across phases
Session 9: 6 common trials across phases
Session 10: 4 common trials across phases
Session 11: 2 common trials across phases
Session 12: 3 common trials across phases
Session 13: 2 common trials across phases
Session 14: 2 common trials across phases
Session 15: 8 common trials across phases
Session 16: 2 common trials across phases
Session 17: 4 common trials across phases
Session 18: 3 common trials across phases
Session 19: 3 common trials across phases
Session 20: 2 common trials across phases
Session 21: 1 common trials across phases
Session 22

In [7]:
import os
import pandas as pd

def update_roi_labels_in_csvs(folder_path, roi_map, file_pattern="subj*_similarity_results.csv"):
    """
    Reads similarity CSV files in a folder, updates ROI labels, and saves them back.

    Parameters:
    - folder_path: folder containing CSVs
    - roi_map: dict mapping 'roi_1' -> 'ERC', 'roi_2' -> '35', etc.
    - file_pattern: glob pattern to match your CSVs (default matches subj*_similarity_results.csv)
    """
    import glob

    csv_files = glob.glob(os.path.join(folder_path, file_pattern))

    for csv_file in csv_files:
        print(f"Processing {csv_file} ...")
        df = pd.read_csv(csv_file)

        # Update ROI column
        df['ROI'] = df['ROI'].map(roi_map).fillna(df['ROI'])

        # Save back (overwrite or save new version)
        df.to_csv(csv_file, index=False)
        print(f"Updated ROI labels saved to {csv_file}")

# ROI mapping dictionary
roi_map = {
    'roi_1': 'ERC',
    'roi_2': '35',
    'roi_3': '36',
    'roi_4': 'SUB',
    'roi_5': 'CA1',
    'roi_6': 'CA2',
    'roi_7': 'CA3',
    'roi_8': 'DG',
    'roi_9': 'PHC',
    'roi_10': 'HT'
}
update_roi_labels_in_csvs("/home/jovyan/cache/memoryNSD", roi_map)


Processing /home/jovyan/cache/memoryNSD/subj08_similarity_results.csv ...
Updated ROI labels saved to /home/jovyan/cache/memoryNSD/subj08_similarity_results.csv
Processing /home/jovyan/cache/memoryNSD/subj01_similarity_results.csv ...
Updated ROI labels saved to /home/jovyan/cache/memoryNSD/subj01_similarity_results.csv
Processing /home/jovyan/cache/memoryNSD/subj02_similarity_results.csv ...
Updated ROI labels saved to /home/jovyan/cache/memoryNSD/subj02_similarity_results.csv
Processing /home/jovyan/cache/memoryNSD/subj03_similarity_results.csv ...
Updated ROI labels saved to /home/jovyan/cache/memoryNSD/subj03_similarity_results.csv
Processing /home/jovyan/cache/memoryNSD/subj04_similarity_results.csv ...
Updated ROI labels saved to /home/jovyan/cache/memoryNSD/subj04_similarity_results.csv
Processing /home/jovyan/cache/memoryNSD/subj05_similarity_results.csv ...
Updated ROI labels saved to /home/jovyan/cache/memoryNSD/subj05_similarity_results.csv
Processing /home/jovyan/cache/memo